*How much money is noise, what shape the noise has, and what follows for anything you measure.*

**You beat the other farm by \$3,000. Does that mean your agent is better?**

It does not, and the number that says so is \$27,300. That is the standard deviation of the margin between the two banks in a Kaggriculture game, measured over 22,702 recorded episodes, which is every episode in the attached corpus counted once rather than once per seat. Against a spread that wide, a three thousand dollar win is most of the way to a coin flip, and half of all games at the top of the ladder are decided by less than \$2,738.

That matters because almost every decision anyone makes here rests on comparing two numbers. Is this version better than that one. Is this opponent hard. Did the change help. **Comparing two numbers is only meaningful once you know the width of the thing they were drawn from**, and that width is not something you can eyeball from a leaderboard.

**So this notebook works out the base distributions of the game and what they cost you.** Where a shape can be derived from how the game works, it is derived and then tested against the corpus, so the test could fail. Where it cannot, it is measured and labelled as measured. Four hypotheses are put up and the data refutes all four; the refutations are kept, in order, because the road to the answer is most of the value and because a shape that merely fits well is not a result.

Everything runs on a public dataset of recorded episodes, so you can change the question and re-run it.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 130,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
    "font.size": 9.5, "axes.titlesize": 11.5, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "legend.frameon": False,
    "lines.linewidth": 1.1, "lines.markersize": 4.0,
    "lines.solid_capstyle": "round", "axes.linewidth": 0.7,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7, "patch.linewidth": 0.7,
})
INK, TEAL, AMBER, GREY, RED = "#0F172A", "#0F766E", "#B45309", "#94A3B8", "#B91C1C"

def find(name):
    for root in ("/kaggle/input", ".", "..", "data"):
        r = Path(root)
        if r.exists():
            for p in r.rglob(name):
                return p
    return None

M = pd.read_parquet(find("matchups_all.parquet"),
                    columns=["episode_id", "opponent_seat", "opponent_rating",
                             "opponent_behaviour", "recorded_bank_yours",
                             "recorded_bank_opponent"])
print(f"{len(M):,} recorded matchups")

## One row per episode, not two

The file carries a row for each side of every episode, so an episode appears twice with its margin written once each way. Anything about the margin has to pick **one seat per episode, at random**, or the symmetry test below is testing that $x$ and $-x$ are mirror images, which is arithmetic rather than evidence.

In [ ]:
rng = np.random.default_rng(17)
E = (M.assign(_r=rng.random(len(M)))
      .sort_values("_r").drop_duplicates("episode_id"))
margin = (E.recorded_bank_yours - E.recorded_bank_opponent).to_numpy()
banks = np.concatenate([E.recorded_bank_yours.to_numpy(),
                        E.recorded_bank_opponent.to_numpy()])
print(f"{len(E):,} independent episodes, {len(banks):,} banks")

# How much money is noise?

Start with the number that decides everything else, and it does not need any assumption about shape at all.

In [ ]:
rho = float(np.corrcoef(E.recorded_bank_yours, E.recorded_bank_opponent)[0, 1])
s_bank = float(banks.std(ddof=1))
s_pred = s_bank * np.sqrt(2 * (1 - rho))
s_obs = float(margin.std(ddof=1))

print(f"  standard deviation of one bank      ${s_bank:>12,.0f}")
print(f"  correlation between the two banks    {rho:>+12.3f}")
print(f"  margin spread PREDICTED by algebra  ${s_pred:>12,.0f}")
print(f"  margin spread observed              ${s_obs:>12,.0f}")
print(f"  error                               ${s_pred - s_obs:>+12,.0f}")
print()
a = np.abs(margin)
for q in (0.05, 0.25, 0.50, 0.75, 0.95):
    print(f"  {int(q*100):>3}% of games are decided by less than ${np.quantile(a, q):>10,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.2))
srt = np.sort(np.abs(margin))
share = np.arange(1, len(srt) + 1) / len(srt)
ax.plot(srt, 100 * share, color=TEAL, label="games decided by less than this")
ax.axvline(s_obs, color=RED, lw=1.0, ls=(0, (4, 3)))
ax.annotate(f"the noise, \\${s_obs:,.0f}", xy=(s_obs, 8), xytext=(6, 0),
            textcoords="offset points", fontsize=9, color=RED)
for amount in (3000, 10000):
    pct = 100 * (np.abs(margin) < amount).mean()
    ax.plot([amount], [pct], marker="o", ms=5, mfc="white", mec=INK, mew=1.2)
    # `pct` is the share of margins SMALLER than `amount`, which is the share a
    # win of that size beats. The complement is the share that beat IT, and
    # labelling the marker with the complement inverts the sentence.
    ax.annotate(f"a \\${amount:,} win beats\\n{pct:.0f} % of margins",
                xy=(amount, pct), xytext=(12, -18), textcoords="offset points",
                fontsize=8.5, color=INK,
                arrowprops=dict(arrowstyle="-", color=GREY, lw=0.7))
ax.set_xscale("log")
ax.set_xlabel("margin, dollars"); ax.set_ylabel("% of games decided by less")
ax.set_title("How big does a win have to be before it means anything?",
             loc="left")
ax.legend(loc="lower right", fontsize=8.5)
plt.tight_layout(); plt.show()

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE ALGEBRA BEHIND IT: Why the spread of a difference is not the sum of the spreads</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>For any two quantities $X$ and $Y$ with standard deviations $s_X$ and $s_Y$ and correlation $\rho$,</p>
$$\operatorname{Var}(X-Y) = s_X^2 + s_Y^2 - 2\rho\, s_X s_Y$$
<p>and when the two have the same spread $s$ this collapses to</p>
$$\operatorname{sd}(X-Y) = s\sqrt{2(1-\rho)}$$
<p>This is algebra, not modelling: it holds whatever shape the banks have, and it needs only that both spreads exist. The reason it is worth writing down here is the $\rho$ term. The two banks in a Kaggriculture game are strongly and positively correlated, because both farms draw from <b>one shared market</b>: a good season for crop prices is a good season for both of you, and a bad shop draw is bad for both.</p>
<p>That correlation is doing you an enormous favour. Without it the margin would have a spread of $s\sqrt{2}$, about 41% wider than a single bank. With $\rho$ near $+0.74$ the factor is roughly \$0.73$ instead, so the margin is <b>narrower</b> than either bank. The shared market cancels most of the season's luck.</p>
<p>It also explains a fact that surprises people the first time they see it: the two banks in a game move together far more than either moves with skill. That is what makes this a common-value game rather than a private-value one, and it is the mechanism behind a rule that sounds wrong out of context, namely that you should prefer wins to money.</p>
<p>The practical use is that you can compute the noise on a difference from quantities you already have, without running a single extra game.</p>

</div>
</details>

---

## The number that ends the argument

The margin between two banks has a spread of about **\$27,300**, and the algebra above predicts it from the banks alone to within a rounding error. Anything smaller than that is inside the noise of a single game.

Worse, it gets tighter exactly where it hurts. The stronger the field you are matched into, the closer the games, while the noise stays where it is.

In [ ]:
BANDS = [(400, 800), (800, 1400), (1400, 2000), (2000, 2400), (2400, 2700),
         (2700, 3200)]
rows = []
for lo, hi in BANDS:
    s = E[(E.opponent_rating >= lo) & (E.opponent_rating < hi)]
    if len(s) < 100:
        continue
    d = (s.recorded_bank_yours - s.recorded_bank_opponent).abs()
    rows.append({"rating band": f"{lo}-{hi}", "games": len(s),
                 "median margin": f"${d.median():,.0f}",
                 "decided by under $5k": f"{100*(d < 5000).mean():.0f} %"})
display(pd.DataFrame(rows))

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.9))
xs, ys = [], []
for lo, hi in BANDS:
    s = E[(E.opponent_rating >= lo) & (E.opponent_rating < hi)]
    if len(s) < 100:
        continue
    xs.append(f"{lo}-{hi}")
    ys.append(float((s.recorded_bank_yours - s.recorded_bank_opponent).abs().median()))
x = np.arange(len(xs))
ax.axhline(s_obs, color=RED, lw=1.0, ls=(0, (4, 3)))
ax.annotate(f"the noise: ${s_obs:,.0f}", xy=(0, s_obs), xytext=(0, 6),
            textcoords="offset points", fontsize=9, color=RED)
ax.plot(x, ys, color=TEAL, marker="o")
ax.fill_between(x, 0, ys, color=TEAL, alpha=0.10, lw=0)
ax.annotate("what a typical game\nis decided by", xy=(x[-2], ys[-2]),
            xytext=(-4, 34), textcoords="offset points", fontsize=9,
            color="#0F766E", ha="right",
            arrowprops=dict(arrowstyle="-", color=GREY, lw=0.7))
ax.set_xticks(x); ax.set_xticklabels(xs, fontsize=8.5)
ax.set_ylim(0, s_obs * 1.18)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.set_xlabel("rating band"); ax.set_ylabel("median absolute margin, dollars")
ax.set_title("The better the field, the more of the result is noise", loc="left")
plt.tight_layout(); plt.show()

At the bottom of the ladder the typical game is decided by most of the noise, \$23,148 against a spread of \$27,300, so a win there carries real information on its own. Near the top the typical game is decided by \$2,738, a tenth of it, and **a single win carries almost no information about which agent is better**. That is not a defect of the ladder, it is what happens when the field converges: everybody finds a similar answer, and the remaining differences are smaller than the weather.

# What shape is a bank?

The spread is settled. The shape decides what you are allowed to do with it, because every interval and every test assumes one.

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; WHAT IS BEHIND THIS: Two mechanisms, two different answers, and only one can be right</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>There is a natural argument for the <b>lognormal</b> and it is worth stating properly, because it is the one most people reach for and this notebook is about to reject it.</p>
<p>Money in this game is not handed to you in independent instalments. It is reinvested: cash buys land and animals, land and animals produce goods, goods sell for cash, cash buys more. Growth proportional to current size with random multiplicative shocks is <b>Gibrat's law</b>, and its limiting distribution is lognormal, not normal. Firm sizes, city populations and income distributions are all lognormal-ish for exactly this reason.</p>
<p>The competing argument is the <b>normal</b>. If a bank is a sum of many roughly independent daily incomes, then the central limit theorem applies and the answer is a normal, no matter what shape a single day's income has.</p>
<p>These are not two ways of saying the same thing. They differ most in the right tail, which is where a "how likely is a bank this large" question lives, and the two make opposite predictions about whether $\log(\text{bank})$ or the bank itself should look symmetric. So this is a fork with a mechanism behind each branch and a test that can settle it, which is what makes it worth running rather than assuming.</p>

</div>
</details>

---

In [ ]:
x = banks[banks > 0]
fits = {}
for name, dist, data, back in (("normal", stats.norm, x, 0.0),
                               ("lognormal", stats.norm, np.log(x),
                                float(np.log(x).sum())),
                               ("gamma", stats.gamma, x, 0.0)):
    par = dist.fit(data, floc=0) if name == "gamma" else dist.fit(data)
    fits[name] = {
        "log-likelihood": float(dist.logpdf(data, *par).sum()) - back,
        "KS p-value": float(stats.kstest(data, dist(*par).cdf).pvalue),
    }
F = pd.DataFrame(fits).T.sort_values("log-likelihood", ascending=False)
display(pd.DataFrame({"log-likelihood": F["log-likelihood"].map("{:,.1f}".format),
                      "KS p-value": F["KS p-value"].map("{:.3g}".format)}))
print(f"skew of log(bank) {stats.skew(np.log(x)):+.2f},  "
      f"excess kurtosis {stats.kurtosis(np.log(x)):+.2f}   "
      f"(both 0 if exactly lognormal)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
for ax, (data, lab) in zip(axes, ((x, "the bank"), (np.log(x), "log of the bank"))):
    osm, osr = stats.probplot(data, dist="norm", fit=False)
    ax.plot(osm, osr, ls="none", marker="o", ms=2.0, mfc=TEAL, mec="none",
            alpha=0.30)
    mu, sd = data.mean(), data.std(ddof=1)
    lo, hi = float(osm.min()), float(osm.max())
    ax.plot([lo, hi], [mu + sd*lo, mu + sd*hi], color=GREY, lw=1.0,
            ls=(0, (4, 3)), label="exactly normal")
    ax.set_title(f"{lab} against a normal", loc="left")
    ax.set_xlabel("normal quantile"); ax.set_ylabel(lab)
    ax.legend(fontsize=8.5)
plt.tight_layout(); plt.show()

## The reinvestment argument is wrong, and it is wrong by a wide margin

**The normal wins on likelihood by thousands of units**, and the log of the bank comes out sharply left-skewed, which is what taking logs of a normal quantity does to it rather than what a lognormal looks like.

Read the p-values with care, and in this direction only. At forty-five thousand observations a goodness-of-fit test rejects **everything**, because it has the power to detect departures far too small to matter to any interval you would ever draw; a p-value of 1e-12 next to one of exactly 0 is not a close call between two live options. What the column is good for is the ordering, and the ordering is not close. The QQ plot is the honest picture of where the normal does fail: in the far tails, and not much anywhere else.

So the mechanism has to be the other one, and there is a reason the multiplicative phase cannot win: **it terminates**. Land and animals are bought in the opening days, after which the farm's productive capacity is fixed and the remaining several hundred turns are additive, harvest and sell and bank. Growth cannot stay proportional while approaching a ceiling, and this game has a ceiling.

That is worth having for its own sake, but its real use is one section down.

# What shape is the margin?

The banks are normal. A difference of two normal quantities is normal. So the margin is normal, and we are done.

It is not, and the gap between those two sentences is the most useful thing in this notebook.

In [ ]:
shapes = {}
for name, dist in (("normal", stats.norm), ("laplace", stats.laplace),
                   ("t", stats.t)):
    par = dist.fit(margin)
    shapes[name] = {
        "log-likelihood": float(dist.logpdf(margin, *par).sum()),
        "KS p-value": float(stats.kstest(margin, dist(*par).cdf).pvalue),
        "df": round(float(par[0]), 2) if name == "t" else None,
    }
S = pd.DataFrame(shapes).T.sort_values("log-likelihood", ascending=False)
display(S)
print(f"skew {stats.skew(margin):+.3f},  "
      f"excess kurtosis {stats.kurtosis(margin):+.2f}   (0 and 0 if normal)")

In [ ]:
# The tails are the whole story, and on a linear axis they are invisible: they
# are rare by construction. A log density is the honest picture.
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2))
lim = float(np.quantile(np.abs(margin), 0.995))
bins = np.linspace(-lim, lim, 81)
mid = 0.5 * (bins[1:] + bins[:-1])
dens, _ = np.histogram(margin, bins=bins, density=True)
pn = stats.norm(*stats.norm.fit(margin)).pdf(mid)
pt_par = stats.t.fit(margin)
pt = stats.t(*pt_par).pdf(mid)

for ax, logy in zip(axes, (False, True)):
    ax.fill_between(mid, dens, color=TEAL, alpha=0.22, lw=0, step="mid",
                    label="the margins actually recorded")
    ax.plot(mid, pn, color=RED, label="the normal that fits them best")
    ax.plot(mid, pt, color=AMBER,
            label=f"a t with {pt_par[0]:.1f} degrees of freedom")
    if logy:
        ax.set_yscale("log")
        ax.set_ylim(max(dens[dens > 0].min() * 0.5, 1e-9), dens.max() * 2)
        ax.set_title("and the tails are far too heavy", loc="left")
    else:
        ax.set_title("the peak is far too sharp for a normal", loc="left")
        ax.legend(fontsize=8.5)
    ax.set_xlabel("margin, dollars")
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
plt.tight_layout(); plt.show()

The normal fails at both ends and it fails visibly. In the left panel the recorded margins pile up at zero three times higher than a normal of the same spread allows, which is the clone effect: a great many games between agents playing nearly the same thing, ending nearly level. In the right panel, with a logarithmic density so that rare events are visible at all, the same fit collapses at the other extreme, where the data still has weight a hundred times past where the normal has effectively none.

A t with about one and a half degrees of freedom tracks both ends. That is a description rather than a mechanism, and the next section is three attempts at the mechanism, all of which fail.

**Worth noting for your own work: the right panel is the one that shows the problem, and it is never the plot anyone draws first.** A distribution's tails are rare by construction, so a linear density hides exactly the region where a rare, expensive event lives.

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; WHAT IS BEHIND THIS: How a difference of two normals fails to be normal</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>The theorem people remember is that a linear combination of normal variables is normal. The condition it needs is stronger than it sounds: the variables must be <b>jointly</b> normal, not merely normal one at a time. Two variables can each be perfectly normal and have a joint distribution that is not, and their difference is then free to be any shape at all.</p>
<p>The standard way this happens is a <b>mixture</b>. Suppose the margin in a game is normal with a spread that depends on which two agents were paired. Pool games from pairings with different spreads and the result is a mixture of normals, which is symmetric, more sharply peaked than a normal, and has heavier tails than a normal. Excess kurtosis of a variance mixture is always positive, never negative, so this is a directional prediction and not a story that fits anything.</p>
<p>That gives a testable claim rather than an explanation. If the tails come from mixing, then dividing each group by <b>its own</b> spread before pooling removes the thing that caused them, and the excess kurtosis should collapse toward zero. If the tails are a property of the margin itself, standardising changes nothing.</p>
<p>The section below runs that test three times, on three different ways of splitting the games, and reports what came back.</p>

</div>
</details>

---

## Three explanations, and the corpus refuses all three

Each hypothesis below says the tails are an artefact of pooling unlike things. Each is tested the same way: standardise inside the groups, pool, and see whether the excess kurtosis goes away.

In [ ]:
def within_kurtosis(groups, floor=30):
    z = [(np.asarray(g, float) - np.mean(g)) / np.std(g, ddof=1)
         for g in groups if len(g) >= floor and np.std(g, ddof=1) > 0]
    return (float(stats.kurtosis(np.concatenate(z))), len(z)) if z else (None, 0)

k0 = float(stats.kurtosis(margin))
E2 = E.assign(margin=E.recorded_bank_yours - E.recorded_bank_opponent,
              weaker=np.minimum(E.recorded_bank_yours, E.recorded_bank_opponent))

# H1  mixing over rating bands
h1 = [g.to_numpy() for _, g in
      E2.assign(b=(E2.opponent_rating // 100) * 100).groupby("b")["margin"]]

# H2  catastrophes: agents that crash or silently do nothing
h2 = [E2[E2.weaker >= t].margin.to_numpy() for t in (0, 10_000, 20_000, 40_000)]

# H3  clones: both sides opening with a byte-identical stream
wide = M.pivot_table(index="episode_id", columns="opponent_seat",
                     values="opponent_behaviour", aggfunc="first").dropna()
wide.columns = ["b0", "b1"]
J = E2.set_index("episode_id").join((wide.b0 == wide.b1).rename("clone"),
                                    how="inner").dropna(subset=["clone"])
h3 = [g.margin.to_numpy() for _, g in J.groupby("clone")]

rows = []
for name, groups in (("H1  mixing over rating bands", h1),
                     ("H2  catastrophic games", h2),
                     ("H3  clones against non-clones", h3)):
    k1, used = within_kurtosis(groups)
    rows.append({"hypothesis": name, "groups": used,
                 "excess kurtosis, pooled": round(k0, 2),
                 "after standardising within": round(k1, 2) if k1 else None,
                 "verdict": "supported" if k1 and k1 < 0.5 * k0 else "REFUTED"})
display(pd.DataFrame(rows))
print(f"\n{100*J.clone.mean():.0f} % of episodes pair two byte-identical openings")
for v, g in J.groupby("clone"):
    print(f"  {'same' if v else 'different'} opening: {len(g):>5} games, "
          f"median absolute margin ${g.margin.abs().median():>9,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.0))

# left: what each hypothesis would have had to do, and what it did
ax = axes[0]
L = pd.DataFrame(rows)
y = np.arange(len(L))
after = [float(v) for v in L["after standardising within"]]
ax.barh(y, after, color=RED, height=0.32)
for yi, v in zip(y, after):
    ax.text(v + 0.7, yi, f"{v:.1f}", va="center", fontsize=9, fontweight="bold")
ax.axvline(k0, color="#475569", lw=0.9, ls=(0, (4, 3)))
ax.axvline(0, color=INK, lw=1.2)
ax.set_yticks(y)
ax.set_yticklabels([h.split("  ")[1] for h in L["hypothesis"]], fontsize=8.5)
ax.set_ylim(len(y) - 0.4, -1.15)
ax.set_xlim(0, max(after) * 1.22)
ax.annotate("if the hypothesis\nwere right", xy=(0, -0.95), xytext=(6, 0),
            textcoords="offset points", fontsize=8.5, color=INK, va="center")
ax.annotate(f"pooled: {k0:.1f}", xy=(k0, -0.95), xytext=(6, 0),
            textcoords="offset points", fontsize=8.5, color="#475569",
            va="center")
ax.annotate("past this line, the split made it WORSE",
            xy=(max(after) * 0.99, len(y) - 0.75), fontsize=8.5, color=RED,
            ha="right", va="center")
ax.set_xlabel("excess kurtosis left after standardising inside the groups")
ax.set_title("Three explanations, and what each one removed", loc="left")

# right: the half of H3 that is right
ax = axes[1]
for v, col, lab in ((True, TEAL, "same opening"), (False, AMBER, "different opening")):
    d = np.abs(J[J.clone == v].margin.to_numpy())
    d = np.sort(d)
    ax.plot(d, 100 * np.arange(1, len(d) + 1) / len(d), color=col,
            label=f"{lab}, n={len(d):,}")
floor = 100 * (J[J.clone].margin == 0).mean()
ax.annotate(f"{floor:.0f} % of clone games end in an EXACT tie,\n"
            f"to the dollar, which is the step at the left",
            xy=(1.6, floor), xytext=(10, 26), textcoords="offset points",
            fontsize=8.5, color=INK,
            arrowprops=dict(arrowstyle="-", color=GREY, lw=0.7))
ax.set_xscale("log")
ax.set_xlabel("margin, dollars"); ax.set_ylabel("% of games decided by less")
ax.set_title("Clones really do end closer, which is the peak but not the tails",
             loc="left")
ax.legend(loc="lower right", fontsize=8.5)
plt.tight_layout(); plt.show()

**None of them.** Splitting by rating band makes it worse. Removing every game in which one side collapsed makes it worse. And the clone hypothesis is the interesting failure: it is **half right and still refuted**.

Games where both agents open with a byte-identical stream really are much closer, by a factor of about five in the typical margin, and that is where the sharp peak at zero comes from. But those same games have the heaviest tails of any group in this notebook. Sharing an opening does not mean sharing an endgame, and whatever produces the tails is not the thing that produces the peak.

So the tails are a property of the margin rather than a property of the pooling. That is an open question, and the honest thing is to leave it open.

## What you do about it does not depend on the answer

At this kurtosis the mean and the standard deviation are poor summaries of a margin, whatever is causing it. Two consequences, and neither needs the mechanism:

**Use the median and a paired sign test, not the mean and a t-test.** The sign test only asks which side of zero each paired difference falls on, so tails cannot move it. A t-test on the same data is being steered by the handful of games that blew out.

**And be careful which spread you divide by.** There is a well-calibrated rule for turning a margin into a win probability, $\Pr[\text{win}] = \Phi(\mu/\sigma)$, and it is defined on the margin against **one** opponent across seeds. The spread of a list with one game against each of forty different opponents is a different quantity: it measures how much opponents differ from each other, and it is far larger. Divide by that one and you have charged your agent for the variance of who it was drawn against.

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; WHAT IS BEHIND THIS: Why the win rule is a probit, and why that is now derived rather than fitted</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>If your margin against a given opponent is normal with mean $\mu$ and standard deviation $\sigma$, then the probability that it lands above zero is</p>
$$\Pr[\text{win}] = \Phi\!\left(\frac{\mu}{\sigma}\right)$$
<p>with $\Phi$ the standard normal cumulative distribution. The quantity being maximised is therefore a <b>signal-to-noise ratio</b>, not an average. Two consequences follow immediately and both are counterintuitive.</p>
<p>First, when you are <b>behind</b>, extra variance helps you. If $\mu < 0$ then increasing $\sigma$ pushes $\mu/\sigma$ toward zero and your win probability toward one half. A change that lowers your typical bank but widens its spread can genuinely raise your chance of winning, and any instinct built on "improve the average" gets that exactly backwards.</p>
<p>Second, when you are <b>ahead</b>, variance is your enemy, and the safe play is the one that narrows the spread even at the cost of the mean.</p>
<p>The reason to trust the probit rather than some other S-shaped curve is the bank result above. The normal is not chosen here because it fits the win rates a little better than a logistic would; on a handful of opponents nothing could distinguish those. It is chosen because a bank was measured to be normal, and a difference of normals against a fixed opponent is normal, so $\Phi$ is what the structure of the game implies. That is a much stronger reason than a fit, and it is available only because the reinvestment argument was tested and rejected first.</p>

</div>
</details>

---

# How many opponents do you actually have?

The same question in a different costume. A pool of recorded opponents has a number of rows, and that number is not how many independent tests it gives you.

In [ ]:
counts = M.opponent_behaviour.value_counts().to_numpy().astype(float)
Ntot = counts.sum()

def expected_distinct(k):
    """E[D(k)] exactly, by inclusion over behaviours. No simulation."""
    from scipy.special import gammaln
    lp = (gammaln(Ntot - counts + 1) - gammaln(Ntot - counts - k + 1)
          - gammaln(Ntot + 1) + gammaln(Ntot - k + 1))
    miss = np.exp(np.where(Ntot - counts - k + 1 > 0, lp, -np.inf))
    return float((1 - miss).sum()), float(np.sqrt((miss * (1 - miss)).sum()))

KS = [10, 20, 40, 80, 160, 320, 640]
rows = [{"you drew": k, "you actually got": round(expected_distinct(k)[0], 1),
         "give or take": round(expected_distinct(k)[1], 1)}
        for k in KS if k <= Ntot]
display(pd.DataFrame(rows))

f1 = int((counts == 1).sum()); f2 = int((counts == 2).sum())
chao1 = len(counts) + (f1*f1/(2*f2) if f2 else f1*(f1-1)/2)
print(f"\n  behaviours observed here          {len(counts):>8,}")
print(f"  seen exactly once                 {f1:>8,}")
print(f"  Good-Turing coverage              {1 - f1/Ntot:>8.3f}")
print(f"  Chao1 lower bound on the field    {chao1:>8,.0f}")
print(f"  so at least this many unseen      {chao1 - len(counts):>8,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.0))

ax = axes[0]
KK = np.unique(np.round(np.logspace(1, np.log10(min(2000, Ntot)), 26)).astype(int))
ED = np.array([expected_distinct(int(k))[0] for k in KK])
SD = np.array([expected_distinct(int(k))[1] for k in KK])
ax.plot(KK, KK, color=GREY, lw=1.0, ls=(0, (4, 3)),
        label="what the size of your pool claims")
ax.fill_between(KK, ED - 1.96 * SD, ED + 1.96 * SD, color=TEAL, alpha=0.16, lw=0)
ax.plot(KK, ED, color=TEAL, label="distinct behaviours you actually drew")
k40 = float(expected_distinct(40)[0])
ax.plot([40], [k40], marker="o", ms=5.5, mfc="white", mec=INK, mew=1.3)
ax.annotate(f"a pool of 40 is {k40:.0f}", xy=(40, k40), xytext=(14, -20),
            textcoords="offset points", fontsize=9, color=INK,
            arrowprops=dict(arrowstyle="-", color=GREY, lw=0.7))
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("opponents drawn"); ax.set_ylabel("distinct behaviours")
ax.set_title("The gap opens at once and never closes", loc="left")
ax.legend(loc="upper left", fontsize=8.5)

ax = axes[1]
spec = pd.Series(counts).value_counts().sort_index()
k = spec.index.to_numpy()[:12]
v = spec.to_numpy()[:12]
ax.bar(k, v, color=[RED if i == 1 else TEAL for i in k], width=0.62)
ax.set_yscale("log")
ax.set_ylim(top=v[0] * 6)
ax.annotate(f"{int(v[0]):,} behaviours seen exactly once,\n"
            f"which is what says the field is\nnot finished arriving",
            xy=(1.35, v[0]), xytext=(30, 4), textcoords="offset points",
            fontsize=8.5, color=INK, va="center",
            arrowprops=dict(arrowstyle="-", color=GREY, lw=0.7))
ax.set_xlabel("times a behaviour appears in the corpus")
ax.set_ylabel("how many behaviours")
ax.set_title("The frequency spectrum, where both estimators come from",
             loc="left")
plt.tight_layout(); plt.show()

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE THEORY BEHIND IT: Two classical estimators that turn a pool into a denominator</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>This is not a Kaggriculture problem. It is the <b>species sampling problem</b>, studied since the 1940s, and it comes with closed forms.</p>
<p>Draw $k$ of the $N$ recorded matchups without replacement. A behaviour with $n_i$ copies is missed entirely with probability $\binom{N-n_i}{k}/\binom{N}{k}$, so the expected number of distinct behaviours you drew is</p>
$$E[D(k)] = \sum_i \left[1 - \frac{\binom{N-n_i}{k}}{\binom{N}{k}}\right]$$
<p>exactly, for every $k$, with no simulation and no resampling. The table above is that formula.</p>
<p>Two further quantities come from the <b>frequency spectrum</b>, meaning how many behaviours were seen exactly once ($f_1$), exactly twice ($f_2$), and so on.</p>
<p><b>Good-Turing coverage</b>, $C = 1 - f_1/n$, estimates the share of the field's probability mass your sample represents. The intuition is that singletons measure the rate at which new behaviour is still arriving: if a large fraction of what you have was seen exactly once, the next opponent you add is about as likely to be new as not, and your sample has not converged.</p>
<p><b>Chao1</b>, $\hat{S} \ge S_{\text{obs}} + f_1^2/(2f_2)$, is a lower bound on how many behaviours exist in the field you drew from. When it sits far above what you observed, the pool has not seen the field, which is a different failure from the pool being small and is not fixed by replaying the same opponents more often.</p>
<p>There is a trap worth naming. "Fraction of seats that share their stream with at least one other seat" is exactly \$1 - f_1/n$ in disguise, and it <b>rises with sample size for a fixed field</b>. Two corpora of different sizes will report very different numbers for the same population, and neither is wrong. Any such figure has to be quoted with the sample size beside it, or it is not comparable to anybody else's, including your own from last week.</p>

</div>
</details>

---

# The control

Everything above measures the game. This measures the data, and it is the only test here that can fail without anything being wrong with your agent.

In [ ]:
a = E[E.opponent_seat == 1]
b = E[E.opponent_seat == 0]
ma = (a.recorded_bank_yours - a.recorded_bank_opponent).to_numpy()
mb = (b.recorded_bank_yours - b.recorded_bank_opponent).to_numpy()
ks = stats.ks_2samp(margin, -margin)
boot = np.array([rng.choice(margin, len(margin), replace=True).mean()
                 for _ in range(2000)])

# EXCLUDE EXACT TIES. A sign test is defined on the non-zero differences, and
# this game produces real ties: two agents running the same code bank the same
# money to the dollar. Counting a tie as "not a win" biases the rate below one
# half and makes a sound corpus look asymmetric.
ties = int((margin == 0).sum())
nz = margin[margin != 0]

print(f"  episodes                 {len(margin):>10,}")
print(f"  exact ties               {ties:>10,}  ({100*ties/len(margin):.2f} %)")
print(f"  mean margin             ${margin.mean():>+10,.0f}")
print(f"  95 % interval           [{np.quantile(boot,.025):>+,.0f}, "
      f"{np.quantile(boot,.975):>+,.0f}]")
print(f"  win rate, ties excluded  {(nz > 0).mean():>10.4f}")
print(f"  sign test p              {stats.binomtest(int((nz>0).sum()), len(nz), 0.5).pvalue:>10.3f}")
print(f"  KS against its mirror    {ks.pvalue:>10.3f}")
print()
print(f"  and counting the ties as losses instead would give "
      f"{(margin > 0).mean():.4f}, p = "
      f"{stats.binomtest(int((margin>0).sum()), len(margin), 0.5).pvalue:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.8))
lim = float(np.quantile(np.abs(margin), 0.98))
bins = np.linspace(-lim, lim, 71)
ax.hist(margin, bins=bins, color=TEAL, alpha=0.28, lw=0, density=True,
        label="the margin")
ax.hist(-margin, bins=bins, histtype="step", color=AMBER, lw=1.1, density=True,
        label="the same data, mirrored")
ax.axvline(0, color=GREY, lw=0.8)
ax.set_xlabel("margin, dollars"); ax.set_ylabel("density")
ax.set_title("A control: a symmetric game must look the same from either seat",
             loc="left")
ax.legend(fontsize=8.5)
plt.tight_layout(); plt.show()

The two seats in this game have the same action set, the same starting endowment and the same payoff rule. So under a seat assignment nobody chose, the recorded margin has to have the same distribution from either side. **Exactly**, not approximately, and with no distributional assumption anywhere.

That makes it a control in the strict sense: a quantity whose value is known in advance if nothing is wrong. If it fails, the fault is in the recording, the replay or the extraction, and not in whatever agent you were about to evaluate. It is the only check here that can tell you your data is broken, and it costs one line.

It passes on this corpus, and the way it nearly did not is the part worth copying.

**Run the same test without excluding exact ties and it fails**, at a win rate of 0.489 and a p-value under 0.001, which would send you looking for a seat bug that is not there. About one game in forty here ends with the two banks equal to the dollar, and **every single one of those is a pairing of two agents that opened identically**. Two copies of the same code, playing the same seed, banking the same money. A sign test is defined on the non-zero differences for exactly this reason, and a control that fires on its own definition is worse than no control, because you will believe it.

# What to take away

**The noise on a Kaggriculture margin is about \$27,300**, and you can compute it from the spread of a bank and the correlation between the two banks without running anything. Anything smaller is inside one game's weather. At the top of the ladder the typical game is decided by \$2,738, a tenth of it.

**A bank is normal.** The reinvestment argument for a lognormal is the natural one and the data rejects it decisively, because the multiplicative phase ends in the opening days and what follows is additive.

**A margin is not**, despite that, and three explanations for its tails were tested and refuted here. Until one survives, use the median and the paired sign test, which the tails cannot move.

**Divide by the right spread.** $\Phi(\mu/\sigma)$ is a statement about one opponent across seeds. The spread across different opponents is a much larger number wearing the same symbol.

**And count your opponents with the frequency spectrum**, not with the length of your table. Any "share of things that appear more than once" figure grows with sample size, so it is meaningless without the sample size printed next to it.

---

## Where this goes next

Every number above is a property of the game rather than of any agent, which makes it a foundation and not a conclusion. Two notebooks stand on it directly:

- [**Six checks before you waste a submission**](https://www.kaggle.com/code/destbreso/six-checks-before-you-waste-a-submission) turns the pool section into something you run: how many opponents your own pool really holds, how many paired games it takes to see a real effect, and what a control on your data looks like.
- [**Kaggriculture Measure Your Agent**](https://www.kaggle.com/code/destbreso/kaggriculture-measure-your-agent) is the harness behind those checks. The reason it refuses to print a win probability from a pool that plays each opponent once is the sigma argument three sections up.

And three that measure the field this one measured the noise of:

- [**Everyone is playing the same opening**](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening), on how far into a game unrelated teams still emit identical actions. That is the mechanism behind the peak at zero here.
- [**The leaderboard has a fossil record**](https://www.kaggle.com/code/destbreso/the-leaderboard-has-a-fossil-record), on lineages sweeping the field and going extinct, which is why any of this has a shelf life.
- [**The leaderboard is a habitat gradient**](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient), on why the band you are matched into changes the answer, which is the rating-band table near the top of this one from the other direction.

---

## References

**Distributions and mechanisms**
- Gibrat, R. (1931). *Les inegalites economiques.* Paris: Sirey.
- Aitchison, J. & Brown, J. A. C. (1957). *The Lognormal Distribution.* Cambridge University Press.
- Limpert, E., Stahel, W. A. & Abbt, M. (2001). Log-normal distributions across the sciences. *BioScience* 51(5), 341-352.

**Species sampling and coverage**
- Good, I. J. (1953). The population frequencies of species and the estimation of population parameters. *Biometrika* 40(3/4), 237-264.
- Chao, A. (1984). Nonparametric estimation of the number of classes in a population. *Scandinavian Journal of Statistics* 11(4), 265-270.
- Chao, A. & Jost, L. (2012). Coverage-based rarefaction and extrapolation. *Ecology* 93(12), 2533-2547.

**Goodness of fit, and its limits at large n**
- Anderson, T. W. & Darling, D. A. (1954). A test of goodness of fit. *JASA* 49(268), 765-769.
- Lilliefors, H. W. (1967). On the Kolmogorov-Smirnov test for normality with mean and variance unknown. *JASA* 62(318), 399-402.

**Robust comparison**
- Wilcoxon, F. (1945). Individual comparisons by ranking methods. *Biometrics Bulletin* 1(6), 80-83.
- Huber, P. J. (1981). *Robust Statistics.* Wiley.
- Wilson, E. B. (1927). Probable inference, the law of succession, and statistical inference. *JASA* 22(158), 209-212.

*Volumes and page numbers are written from memory and are worth checking before quoting; the titles and years are the load-bearing part.*